# HelpersJ

> Helper functions for CRLD agents (JAX variants with J suffix).

In [ ]:
#| default_exp Utils/HelpersJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import jax
import jax.numpy as jnp
from jax import jit
from functools import partial

In [ ]:
#| export
def make_variable_vectorJ(
    variable,  # can be iterable or float or int
    length: int  # length of the vector
):  # vector
    "Turn a `variable` into a vector or check that `length` is consistent."
    if hasattr(variable, '__iter__'):
        assert len(variable) == length, 'Wrong number given'
        return jnp.array(variable, dtype=jnp.float32)
    else:
        return jnp.repeat(jnp.array(variable, dtype=jnp.float32), length)

In [ ]:
#| export
@jit
def compute_stationarydistributionJ(Tkk: jnp.ndarray):  # Transition matrix
    """Compute stationary distribution for transition matrix `Tkk`."""
    oeival, oeivec = jnp.linalg.eig(Tkk.T)
    oeival = oeival.real
    oeivec = oeivec.real

    get_mask = lambda tol: jnp.abs(oeival - 1) < tol

    tolerances = jax.lax.map(lambda x: 0.1 ** x, jnp.arange(1, 16, 1))
    masks = jax.lax.map(get_mask, tolerances)
    ix = jnp.max(jnp.where(masks.sum(-1) >= 1, jnp.arange(len(masks)), -1))
    mask = masks[ix]
    tol = tolerances[ix]

    meivec = jnp.where(mask, oeivec, -42)

    dist = meivec / meivec.sum(axis=0, keepdims=True)
    dist = jnp.where(dist < tol, 0, dist)
    dist = dist / dist.sum(axis=0, keepdims=True)

    return jnp.where(meivec == -42, -10, dist)

## Tests

In [ ]:
# Test make_variable_vectorJ
v = make_variable_vectorJ(0.9, 3)
assert v.shape == (3,)
assert jnp.allclose(v, jnp.array([0.9, 0.9, 0.9]))

v2 = make_variable_vectorJ([0.1, 0.5, 0.9], 3)
assert v2.shape == (3,)
assert jnp.allclose(v2, jnp.array([0.1, 0.5, 0.9]))
print('make_variable_vectorJ: OK')

In [ ]:
# Test compute_stationarydistributionJ on a simple 2-state chain
T = jnp.array([[0.8, 0.2],
               [0.3, 0.7]])
pS = compute_stationarydistributionJ(T)
# Find valid columns (not -10)
valid = pS.mean(0) != -10
pS_valid = pS[:, valid]
assert pS_valid.shape[1] >= 1
# Should sum to 1
assert jnp.allclose(pS_valid[:, 0].sum(), 1.0, atol=1e-5)
print('compute_stationarydistributionJ: OK')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()